# CPU Benchmark Results Analysis
This notebook analyzes the results from the SmartSim, AIxelerator, and PhyDLL benchmarking runs.
Each model is plotted in its own distinct graph.
Different colors are used for `BIND` vs `NOBIND` configurations.
The original ordering of configurations in the data is preserved.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv('provider_results.csv')

# Clean data
df_all = df.copy()
df = df[df['status'] == 'SUCCESS'].copy()
df['time_s'] = pd.to_numeric(df['time_s'], errors='coerce')
df['max_rss_mb'] = pd.to_numeric(df['max_rss_mb'], errors='coerce')

# Calculate Success/Fail Rates
df_all['is_success'] = (df_all['status'] == 'SUCCESS')
success_rates = df_all.groupby(['model', 'provider']).agg(
    total_runs=('status', 'count'),
    successes=('is_success', 'sum')
).reset_index()
success_rates['fail_rate'] = 100 * (1 - success_rates['successes'] / success_rates['total_runs'])
print("Failure Rates by Model and Provider:")
display(success_rates.sort_values('fail_rate', ascending=False))

# Drop NA
df = df.dropna(subset=['time_s', 'max_rss_mb'])

# We keep the original order as requested by the user.
df['original_order'] = range(len(df))

# Feature engineering for colors
def get_bind_type(provider):
    if 'NOBIND' in provider:
        return 'NOBIND'
    elif 'BIND' in provider:
        return 'BIND'
    else:
        return 'OTHER'

df['bind_type'] = df['provider'].apply(get_bind_type)

# Calculate statistics
summary = df.groupby(['model', 'provider'], sort=False).agg(
    median_time=('time_s', 'median'),
    mean_time=('time_s', 'mean'),
    var_time=('time_s', 'var'),
    std_time=('time_s', 'std'),
    count=('time_s', 'count')
).reset_index()

# 95% CI calculation (approximate using 1.96 * std / sqrt(N))
summary['95_ci_lower'] = summary['mean_time'] - 1.96 * summary['std_time'] / np.sqrt(summary['count'])
summary['95_ci_upper'] = summary['mean_time'] + 1.96 * summary['std_time'] / np.sqrt(summary['count'])

summary.to_csv('summary_statistics.csv', index=False)
display(summary.head())


In [ ]:
models = df['model'].unique()

colors = {'BIND': '#1f77b4', 'NOBIND': '#ff7f0e', 'OTHER': '#2ca02c'}

for model in models:
    model_df = df[df['model'] == model].copy()
    
    ordered_providers = model_df['provider'].drop_duplicates().tolist()
    
    plot_data = []
    for prov in ordered_providers:
        prov_df = model_df[model_df['provider'] == prov]
        mean_val = prov_df['time_s'].mean()
        std_val = prov_df['time_s'].std()
        bind_type = prov_df['bind_type'].iloc[0]
        count = len(prov_df)
        ci = 1.96 * std_val / np.sqrt(count) if count > 1 else 0
        plot_data.append({
            'provider': prov,
            'mean': mean_val,
            'ci': ci,
            'bind_type': bind_type
        })
        
    plot_df = pd.DataFrame(plot_data)
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(plot_df['provider'], plot_df['mean'], yerr=plot_df['ci'], capsize=5)
    
    for i, bar in enumerate(bars):
        bar.set_color(colors[plot_df.iloc[i]['bind_type']])
        
    plt.title(f"Execution Time for Model: {model}")
    plt.ylabel("Time (seconds)")
    plt.xlabel("Provider Configuration")
    plt.xticks(rotation=45, ha='right')
    
    import matplotlib.patches as mpatches
    legend_patches = [mpatches.Patch(color=color, label=label) for label, color in colors.items()]
    plt.legend(handles=legend_patches, title="Binding Type")
    
    plt.tight_layout()
    plt.show()
